In [ ]:
!pip install gspread oauth2client geopy pandas -q

import gspread
from google.colab import auth
from google.auth import default
import pandas as pd
from geopy.distance import geodesic
from datetime import datetime

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("Authentication Complete\n")

SPREADSHEET_NAME = "MD VRP 500"  # Input File

CUSTOMERS_SHEET = "Customers"  # Sheet Names
LOCKERS_SHEET = "Lockers"
DEPOT_SHEET = "Depot"

OUTPUT_SHEET = "Complete_Distance_Matrix"

print("=" * 60)
print("Loading Data from Google Sheets")
print("=" * 60)

start_time = datetime.now()

spreadsheet = gc.open(SPREADSHEET_NAME)

customers_ws = spreadsheet.worksheet(CUSTOMERS_SHEET)
customers_data = customers_ws.get_all_records()
df_customers = pd.DataFrame(customers_data)

lockers_ws = spreadsheet.worksheet(LOCKERS_SHEET)
lockers_data = lockers_ws.get_all_records()
df_lockers = pd.DataFrame(lockers_data)

depot_ws = spreadsheet.worksheet(DEPOT_SHEET)
depot_data = depot_ws.get_all_records()
df_depot = pd.DataFrame(depot_data)

print(f"Customers loaded: {len(df_customers)}")
print(f"Lockers loaded: {len(df_lockers)}")
print(f"Depot loaded: {len(df_depot)}")

DEPOT_ID = df_depot.iloc[0]['Depot_ID']
DEPOT_LAT = df_depot.iloc[0]['Latitude']
DEPOT_LON = df_depot.iloc[0]['Longitude']

print(f"\nDepot: {DEPOT_ID} at ({DEPOT_LAT}, {DEPOT_LON})")

total_locations = 1 + len(df_customers) + len(df_lockers)
total_calculations = total_locations * (total_locations - 1)

print(f"\nTotal locations: {total_locations}")
print(f"Expected calculations: {total_calculations:,}")

def calculate_distance(lat1, lon1, lat2, lon2):
    """Calculate Geodesic Distance"""
    point1 = (lat1, lon1)
    point2 = (lat2, lon2)
    distance = geodesic(point1, point2).kilometers
    return round(distance, 4)

all_distances = []

all_locations = []

all_locations.append({
    'ID': DEPOT_ID,
    'Type': 'Depot',
    'Latitude': DEPOT_LAT,
    'Longitude': DEPOT_LON
})

for _, row in df_customers.iterrows():
    all_locations.append({
        'ID': row['Customer_ID'],
        'Type': 'Customer',
        'Latitude': row['Latitude'],
        'Longitude': row['Longitude']
    })

for _, row in df_lockers.iterrows():
    all_locations.append({
        'ID': row['Locker_ID'],
        'Type': 'Locker',
        'Latitude': row['Latitude'],
        'Longitude': row['Longitude']
    })

print(f"Combined {len(all_locations)} locations\n")

print("=" * 60)
print("Calculating All Pairwise Distances")
print("=" * 60)

count = 0
for i, loc1 in enumerate(all_locations):
    for j, loc2 in enumerate(all_locations):
        if i != j:
            distance = calculate_distance(
                loc1['Latitude'], loc1['Longitude'],
                loc2['Latitude'], loc2['Longitude']
            )

            all_distances.append({
                'From_ID': loc1['ID'],
                'To_ID': loc2['ID'],
                'Distance_km': distance
            })

            count += 1

print(f"\nCalculated {count:,} distance pairs\n")

print("=" * 60)
print("Creating Distance Matrix")
print("=" * 60)

df_distances = pd.DataFrame(all_distances)

print(f"Total distances: {len(df_distances):,}")
print(f"Expected: {total_calculations:,}")

if len(df_distances) == total_calculations:
    print(f"All distances calculated\n")
else:
    print(f"Mismatch detected\n")

print("=" * 60)
print("Distance Statistics")
print("=" * 60)

print(f"  Min distance: {df_distances['Distance_km'].min():.4f} km")
print(f"  Max distance: {df_distances['Distance_km'].max():.4f} km")
print(f"  Mean distance: {df_distances['Distance_km'].mean():.4f} km")
print()


print("=" * 60)
print("Saving to Google Sheets")
print("=" * 60)

try:
    existing = spreadsheet.worksheet(OUTPUT_SHEET)
    spreadsheet.del_worksheet(existing)
    print(f"Deleted existing '{OUTPUT_SHEET}'")
except:
    pass

print(f"Creating new sheet '{OUTPUT_SHEET}'")
new_ws = spreadsheet.add_worksheet(
    title=OUTPUT_SHEET,
    rows=min(len(df_distances) + 100, 200000),
    cols=5
)

print(f"Writing {len(df_distances):,} rows to Google Sheets")
headers = df_distances.columns.tolist()
values = [headers] + df_distances.values.tolist()
new_ws.update('A1', values)

new_ws.format('A1:C1', {
    "textFormat": {"bold": True},
    "backgroundColor": {"red": 0.9, "green": 0.9, "blue": 0.9}
})

print(f"Distance matrix saved to '{OUTPUT_SHEET}'\n")

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print("=" * 60)
print("DISTANCE MATRIX COMPLETE!")
print("=" * 60)

print(f"\n Summary:")
print(f"  • Total locations: {len(all_locations)}")
print(f"  • Total distances: {len(df_distances):,}")
print(f"  • Method: Geodesic")
print(f"  • Processing time: {duration/60:.1f} minutes")

print(f"\nGoogle Sheets updated:")
print(f"  • File: '{SPREADSHEET_NAME}'")
print(f"  • Sheet: '{OUTPUT_SHEET}'")